In [ ]:
# change heic to jpg
import os
from pathlib import Path
from PIL import Image
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    print("can not import pillow-heif, installing...")
    import sys
    !{sys.executable} -m pip install pillow-heif
    import pillow_heif
    pillow_heif.register_heif_opener()

def convert_heic_to_jpg(src_dir, dst_dir=None):
    """
    Convert all .heic images in src_dir to .jpg format and save them to dst_dir (overwrite src_dir if not specified).
    """
    src_dir = Path(src_dir)
    dst_dir = Path(dst_dir) if dst_dir else src_dir
    dst_dir.mkdir(parents=True, exist_ok=True)
    heic_files = list(src_dir.glob('*.heic')) + list(src_dir.glob('*.HEIC'))
    print(f"Found {len(heic_files)} HEIC images.")
    for heic_path in heic_files:
        img = Image.open(heic_path)
        jpg_path = dst_dir / (heic_path.stem + '.jpg')
        img.convert('RGB').save(jpg_path, 'JPEG')
        print(f"Converted: {heic_path.name} -> {jpg_path.name}")
    print("All conversions completed!")

def main():
    convert_heic_to_jpg(r"C:\Users\zeyua\Downloads\ChiWah-20260104T081553Z-3-001\ChiWah")
    
if __name__ == "__main__":
    main()
    

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path
from PIL import Image

def auto_label_images(
    image_dir,
    model_path,
    output_dir,
    conf_threshold=0.25,
    iou_threshold=0.7
):
    """
    use YOLO model to automatically label images in a directory.

    parameters:
        image_dir: directory containing images
        model_path: path to YOLO model (.pt file)
        output_dir: directory to save output label files
        conf_threshold: confidence threshold (default 0.25)
        iou_threshold: IOU threshold (default 0.7)
    """

    # Create output directories
    labels_dir = os.path.join(output_dir, 'labels')
    images_dir = os.path.join(output_dir, 'images')
    os.makedirs(labels_dir, exist_ok=True)
    os.makedirs(images_dir, exist_ok=True)

    # load YOLO model
    print(f"Loading model: {model_path}")
    model = YOLO(model_path)

    # Supported image formats
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp', '.heic']

    # get list of image files
    image_files = []
    for ext in image_extensions:
        image_files.extend(Path(image_dir).glob(f'*{ext}'))
        image_files.extend(Path(image_dir).glob(f'*{ext.upper()}'))

    print(f"Found {len(image_files)} images to process.")

    # Process each image
    for idx, image_path in enumerate(image_files, 1):
        print(f"Processing [{idx}/{len(image_files)}]: {image_path.name}")

        # Run inference
        results = model.predict(
            source=str(image_path),
            conf=conf_threshold,
            iou=iou_threshold,
            verbose=False
        )

        # get image dimensions
        img = Image.open(image_path)
        img_width, img_height = img.size

        # create label file path
        label_filename = image_path.stem + '.txt'
        label_path = os.path.join(labels_dir, label_filename)

        # save label file (YOLO format)
        with open(label_path, 'w') as f:
            for result in results:
                boxes = result.boxes
                for box in boxes:
                    # get class ID
                    class_id = int(box.cls[0])

                    # get bounding box coordinates (xyxy format)
                    x1, y1, x2, y2 = box.xyxy[0].tolist()

                    # convert to YOLO format (x_center, y_center, width, height) normalized coordinates
                    x_center = ((x1 + x2) / 2) / img_width
                    y_center = ((y1 + y2) / 2) / img_height
                    width = (x2 - x1) / img_width
                    height = (y2 - y1) / img_height

                    # write to label file
                    f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

        # copy image to output directory
        import shutil
        shutil.copy2(image_path, os.path.join(images_dir, image_path.name))

        # show number of detections
        num_detections = len(results[0].boxes) if results else 0
        print(f"  Detected {num_detections} objects")

    print("\nLabeling completed!")
    print(f"Label files saved in: {labels_dir}")
    print(f"Image files saved in: {images_dir}")
    print("\nYou can upload the contents of these two folders to Roboflow for viewing and further annotation")

    # Save class names file (optional, for reference)
    classes_path = os.path.join(output_dir, 'classes.txt')
    with open(classes_path, 'w', encoding='utf-8') as f:
        for idx, name in model.names.items():
            f.write(f"{idx}: {name}\n")
    print(f"Class names saved in: {classes_path}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
if __name__ == "__main__":
    # ========== Configuration Parameters ==========

    # Image directory path (folder containing images to be labeled)
    IMAGE_DIR = "C:\\Users\\zeyua\\Downloads\\dt"

    # YOLO model path
    MODEL_PATH = "C:\\Users\\zeyua\\Downloads\\HKU-Library-Booking-System-with-Real-time-Space-Monitoring-for-iOS-and-Android\\computer_vision\\train_models\\yolo11x.pt"

    # Output directory path
    OUTPUT_DIR = "C:\\Users\\zeyua\\Downloads\\HKU-Library-Booking-System-with-Real-time-Space-Monitoring-for-iOS-and-Android\\computer_vision\\train_models\\output"  # Labeling results output path

    # Confidence threshold (between 0 and 1, higher is more strict)
    CONF_THRESHOLD = 0.25

    # IOU threshold (for non-maximum suppression)
    IOU_THRESHOLD = 0.7

    # ==============================

    # Execute automatic labeling
    auto_label_images(
        image_dir=IMAGE_DIR,
        model_path=MODEL_PATH,
        output_dir=OUTPUT_DIR,
        conf_threshold=CONF_THRESHOLD,
        iou_threshold=IOU_THRESHOLD
    )